In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

In [2]:
# =========================
# Step 0) The parameters
# =========================
PATH = "../../customers_with_segments.csv"

FEATURES = [
    "MT_TTC_NET_total_spend",
    "CD_TICKET_UNIQUE_total_transactions",
    "avg_basket_value",
    "avg_items_per_basket",
    "product_diversity_score",
    "promo_percentage",
    "QTY_PROMO_STORE_store_promo_qty",
    "QTY_PROMO_NAT_nat_promo_qty",
    "ID_MAG_TIERS_stores_visited",
    "store_loyalty_score",
    "recency_days",
    "avg_days_between_visits",
]
TARGET = "Cluster"

# The threshold 
MAX_ANT_LEN = 2          # maximum antecedent（start with 2, easy and fast to explain）
MIN_SUPPORT_JOINT = 0.01 # support(A ∧ cluster) >= 1%
MIN_CONFIDENCE = 0.50
MIN_LIFT = 1.30
TOP_N_PER_CLUSTER = 10

In [4]:
# =========================
# Step 1) Data read
# =========================
df = pd.read_csv(PATH)

print("Shape:", df.shape)
print("Columns:", len(df.columns))

missing_cols = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing_cols:
    raise ValueError(f"missing cols：{missing_cols}")

df_sel = df[FEATURES + [TARGET]].copy()

# numerical
for c in FEATURES:
    df_sel[c] = pd.to_numeric(df_sel[c], errors="coerce")
df_sel[TARGET] = df_sel[TARGET].astype(int)

print("\nMissing values per selected column (top 10):")
print(df_sel.isna().sum().sort_values(ascending=False).head(10))

print("\nCluster distribution:")
print(df_sel[TARGET].value_counts(normalize=True).sort_index().round(4))

Shape: (121112, 28)
Columns: 28

Missing values per selected column (top 10):
MT_TTC_NET_total_spend                 0
CD_TICKET_UNIQUE_total_transactions    0
avg_basket_value                       0
avg_items_per_basket                   0
product_diversity_score                0
promo_percentage                       0
QTY_PROMO_STORE_store_promo_qty        0
QTY_PROMO_NAT_nat_promo_qty            0
ID_MAG_TIERS_stores_visited            0
store_loyalty_score                    0
dtype: int64

Cluster distribution:
Cluster
0    0.2898
1    0.5186
2    0.1110
3    0.0805
Name: proportion, dtype: float64


In [5]:
# =========================
# Step 2) Three bins（low/mid/high） -> generate “label items”
# =========================
def bin_3(s: pd.Series) -> pd.Series:

    s = s.astype(float)
    try:
        b = pd.qcut(s, q=3, labels=["low", "mid", "high"], duplicates="drop")
        # if works, stick on it；or fallback
        if hasattr(b, "cat") and len(b.cat.categories) == 3:
            return b.astype(str)
    except Exception:
        pass

    # fallback：with 1/3 与 2/3 to cut
    t1, t2 = s.quantile(1/3), s.quantile(2/3)
    lab = np.where(s <= t1, "low", np.where(s <= t2, "mid", "high"))
    return pd.Series(lab, index=s.index, dtype="object")


# feature(conlum) => label（row for item）
binned = pd.DataFrame({
    c: bin_3(df_sel[c]).map(lambda x, c=c: f"{c}={x}")
    for c in FEATURES
})
binned["cluster_item"] = df_sel[TARGET].map(lambda k: f"cluster={k}")

print("\nBin distribution check (each feature shows top 3 bins):")
for c in FEATURES[:6]:
    print(f"- {c}:")
    print(binned[c].value_counts(normalize=True).head(3).round(4).to_string())
print("...")



Bin distribution check (each feature shows top 3 bins):
- MT_TTC_NET_total_spend:
MT_TTC_NET_total_spend
MT_TTC_NET_total_spend=low     0.3334
MT_TTC_NET_total_spend=high    0.3333
MT_TTC_NET_total_spend=mid     0.3333
- CD_TICKET_UNIQUE_total_transactions:
CD_TICKET_UNIQUE_total_transactions
CD_TICKET_UNIQUE_total_transactions=low     0.732
CD_TICKET_UNIQUE_total_transactions=high    0.268
- avg_basket_value:
avg_basket_value
avg_basket_value=low     0.3335
avg_basket_value=mid     0.3333
avg_basket_value=high    0.3332
- avg_items_per_basket:
avg_items_per_basket
avg_items_per_basket=low     0.3333
avg_items_per_basket=high    0.3333
avg_items_per_basket=mid     0.3333
- product_diversity_score:
product_diversity_score
product_diversity_score=low     0.3377
product_diversity_score=mid     0.3342
product_diversity_score=high    0.3282
- promo_percentage:
promo_percentage
promo_percentage=low     0.6887
promo_percentage=high    0.3113
...


In [6]:
# =========================
# Step 3) Encode each row’s set of labels into a bitset (for computational efficiency)
# =========================
items = sorted(pd.unique(binned[FEATURES].values.ravel()))
print("\nTotal unique items:", len(items))

# bitset -> uint64
if len(items) > 63:
    raise ValueError(
        f"items={len(items)} to much（>63），please decrese FEATURES or change the coding method"
    )

item_to_bit = {it: i for i, it in enumerate(items)}
N = len(binned)
bits = np.zeros(N, dtype=np.uint64)
one = np.uint64(1)

for col in FEATURES:
    idx = binned[col].map(item_to_bit).to_numpy(dtype=np.uint64)
    bits |= (one << idx)

clusters = sorted(df_sel[TARGET].unique())
cluster_masks = {k: (df_sel[TARGET].to_numpy() == k) for k in clusters}
p_cluster = {k: cluster_masks[k].mean() for k in clusters}


Total unique items: 31


In [7]:
# =========================
# Step 4) Enumerate antecedents (1–2 items) and directly compute support, confidence, and lift
# =========================
def mine_rules(max_len=2, min_support_joint=0.01, min_conf=0.5, min_lift=1.3):
    rules = []
    item_bits = [(it, (np.uint64(1) << np.uint64(i))) for it, i in item_to_bit.items()]
    item_bits = sorted(item_bits, key=lambda x: x[0])

    for r in range(1, max_len + 1):
        for combo in combinations(item_bits, r):
            ant_items = [c[0] for c in combo]
            ant_bits = np.uint64(0)
            for _, b in combo:
                ant_bits |= b

            presence = (bits & ant_bits) == ant_bits  # if A makes sense
            suppA = presence.mean()

            if suppA < min_support_joint:
                continue

            for k in clusters:
                supp_joint = (presence & cluster_masks[k]).mean()  # support(A∧cluster=k)
                if supp_joint < min_support_joint:
                    continue

                conf = supp_joint / suppA
                if conf < min_conf:
                    continue

                lift = conf / p_cluster[k]
                if lift < min_lift:
                    continue

                rules.append({
                    "cluster": k,
                    "antecedent": " & ".join(ant_items),
                    "consequent": f"cluster={k}",
                    "support": supp_joint,
                    "confidence": conf,
                    "lift": lift,
                    "support_antecedent": suppA,
                })

    return pd.DataFrame(rules)

rules_df = mine_rules(
    max_len=MAX_ANT_LEN,
    min_support_joint=MIN_SUPPORT_JOINT,
    min_conf=MIN_CONFIDENCE,
    min_lift=MIN_LIFT
)

print("\nTotal rules found:", len(rules_df))
if len(rules_df) == 0:
    print("No rules were found: please relax the thresholds (e.g., lower MIN_SUPPORT_JOINT, MIN_CONFIDENCE, or MIN_LIFT).")


Total rules found: 191


In [15]:
# Step 5) Output the Top-N rules for each cluster and save the results
# =========================
out = []
for k in clusters:
    sub = (rules_df[rules_df["cluster"] == k]
           .sort_values(["lift", "confidence", "support"], ascending=[False, False, False])
           .head(TOP_N_PER_CLUSTER))
    out.append(sub)

result = pd.concat(out, ignore_index=True) if out else rules_df

print("\nTop rules per cluster (preview):")
cols_show = ["cluster", "antecedent", "support", "confidence", "lift"]
print(result[cols_show].to_string(index=False))

# Save the outcome
result.to_csv("cluster_explain_rules_top.csv", index=False)
print("\nSaved: cluster_explain_rules_top.csv")



Top rules per cluster (preview):
 cluster                                                              antecedent  support  confidence     lift
       0                   MT_TTC_NET_total_spend=low & store_loyalty_score=high 0.203184    0.998701 3.446200
       0      CD_TICKET_UNIQUE_total_transactions=low & store_loyalty_score=high 0.262154    0.994861 3.432949
       0                  avg_days_between_visits=low & store_loyalty_score=high 0.263516    0.994640 3.432184
       0                            recency_days=high & store_loyalty_score=high 0.106174    0.993357 3.427756
       0                  product_diversity_score=low & store_loyalty_score=high 0.250974    0.992749 3.425661
       0                     avg_items_per_basket=low & store_loyalty_score=high 0.178108    0.992135 3.423541
       0                         avg_basket_value=low & store_loyalty_score=high 0.189956    0.990315 3.417260
       0                         promo_percentage=low & store_loyalty_score=hi

In [16]:
import numpy as np
import pandas as pd
from itertools import combinations

# =========================
# Extra Block) Re-run ONLY for rule-poor clusters using 2-item antecedents
# with relaxed thresholds (preferred vs jumping to 3-item)
# =========================

# 1) Identify "rule-poor" clusters under the original 1~2-item mining
MIN_RULES_PER_CLUSTER = 6
counts = rules_df.groupby("cluster").size().reindex(clusters, fill_value=0)
poor_clusters = counts[counts < MIN_RULES_PER_CLUSTER].index.tolist()

print("Rules per cluster (original 1~2 items):")
print(counts.to_string())
print("\nRule-poor clusters to supplement with relaxed 2-item rules:", poor_clusters)

if len(poor_clusters) == 0:
    print("\nNo supplementation needed.")
else:
    # 2) Relaxed thresholds ONLY for these clusters (2-item rules)
    MIN_SUPPORT_JOINT_2_RELAX = 0.005   # e.g., 0.5% joint support
    MIN_CONFIDENCE_2_RELAX   = 0.40
    MIN_LIFT_2_RELAX         = 1.30

    # Prepare item->bit list once
    item_bits = [(it, (np.uint64(1) << np.uint64(i))) for it, i in item_to_bit.items()]
    item_bits = sorted(item_bits, key=lambda x: x[0])

    relaxed_rules = []
    r = 2  # ONLY 2-item antecedents

    for combo in combinations(item_bits, r):
        ant_items = [c[0] for c in combo]
        ant_bits = np.uint64(0)
        for _, b in combo:
            ant_bits |= b

        presence = (bits & ant_bits) == ant_bits
        suppA = presence.mean()

        # small pruning: if antecedent itself is rarer than joint threshold, skip
        if suppA < MIN_SUPPORT_JOINT_2_RELAX:
            continue

        # Only compute for poor clusters
        for k in poor_clusters:
            supp_joint = (presence & cluster_masks[k]).mean()
            if supp_joint < MIN_SUPPORT_JOINT_2_RELAX:
                continue

            conf = supp_joint / suppA
            if conf < MIN_CONFIDENCE_2_RELAX:
                continue

            lift = conf / p_cluster[k]
            if lift < MIN_LIFT_2_RELAX:
                continue

            relaxed_rules.append({
                "cluster": k,
                "antecedent": " & ".join(ant_items),
                "consequent": f"cluster={k}",
                "support": supp_joint,
                "confidence": conf,
                "lift": lift,
                "support_antecedent": suppA,
                "antecedent_len": 2
            })

    relaxed_df = pd.DataFrame(relaxed_rules)
    print("\nRelaxed 2-item rules found (only for poor clusters):", len(relaxed_df))

    # 3) Save these relaxed rules separately (recommended)
    relaxed_df.to_csv("cluster_rules_2item_relaxed_only_poor_clusters.csv", index=False)
    print("Saved: cluster_rules_2item_relaxed_only_poor_clusters.csv")

    # 4) Build a Top-N table for poor clusters from relaxed rules
    TOP_N_PER_CLUSTER = TOP_N_PER_CLUSTER  # reuse your existing variable

    top_relaxed = []
    for k in poor_clusters:
        sub = (relaxed_df[relaxed_df["cluster"] == k]
               .sort_values(["lift", "confidence", "support"], ascending=[False, False, False])
               .head(TOP_N_PER_CLUSTER))
        top_relaxed.append(sub)

    top_relaxed_df = pd.concat(top_relaxed, ignore_index=True) if len(top_relaxed) else relaxed_df
    top_relaxed_df.to_csv("cluster_explain_rules_top_2item_relaxed.csv", index=False)
    print("Saved: cluster_explain_rules_top_2item_relaxed.csv")

    # Optional preview
    if len(top_relaxed_df) > 0:
        print("\nPreview (relaxed 2-item top rules for poor clusters):")
        print(top_relaxed_df[["cluster","antecedent","support","confidence","lift"]].to_string(index=False))

Rules per cluster (original 1~2 items):
cluster
0     80
1    101
2      8
3      2

Rule-poor clusters to supplement with relaxed 2-item rules: [3]

Relaxed 2-item rules found (only for poor clusters): 10
Saved: cluster_rules_2item_relaxed_only_poor_clusters.csv
Saved: cluster_explain_rules_top_2item_relaxed.csv

Preview (relaxed 2-item top rules for poor clusters):
 cluster                                              antecedent  support  confidence      lift
       3      MT_TTC_NET_total_spend=high & avg_basket_value=low 0.008455    0.998051 12.391175
       3     avg_basket_value=low & product_diversity_score=high 0.011593    0.791878  9.831466
       3 avg_items_per_basket=low & product_diversity_score=high 0.009859    0.766367  9.514737
       3  MT_TTC_NET_total_spend=high & avg_items_per_basket=low 0.009644    0.689086  8.555257
       3      MT_TTC_NET_total_spend=high & avg_basket_value=mid 0.039558    0.535606  6.649756
       3     QTY_PROMO_NAT_nat_promo_qty=high & recenc

In [17]:
# keep clusters 0/1/2 from original result, replace cluster 3 with relaxed rules
non_poor = [0,1,2]
merged = pd.concat([
    result[result["cluster"].isin(non_poor)].copy(),
    top_relaxed_df.copy()   # 你补跑代码里的 top_relaxed_df
], ignore_index=True)

merged.to_csv("cluster_explain_rules_top_final.csv", index=False)

In [14]:
# # Just added for the cluster 3


# # =========================
# # Extra Block) Re-run ONLY for clusters with few rules (3-item antecedents)
# # (Minimal change: keep the original 1~2-item mining intact)
# # =========================
# import numpy as np
# import pandas as pd
# from itertools import combinations

# # 1) Decide which clusters are "rule-poor"
# MIN_RULES_PER_CLUSTER = 6  # e.g., if a cluster has <6 rules from 1~2 items, we will supplement with 3-item rules

# counts_2 = rules_df.groupby("cluster").size().reindex(clusters, fill_value=0)
# poor_clusters = counts_2[counts_2 < MIN_RULES_PER_CLUSTER].index.tolist()

# print("Rules per cluster (1~2 items):")
# print(counts_2.to_string())
# print("\nRule-poor clusters to supplement:", poor_clusters)

# # If no cluster needs supplementation, skip
# if len(poor_clusters) == 0:
#     print("\nNo supplementation needed.")
# else:
#     # 2) Mine 3-item antecedents only (to avoid duplicating 1~2-item rules)
#     #    Use slightly relaxed thresholds (optional but usually necessary)
#     MIN_SUPPORT_JOINT_3 = 0.005   # joint support(A∧cluster) for 3-item (often needs to be lower)
#     MIN_CONFIDENCE_3   = 0.40
#     MIN_LIFT_3         = 1.30

#     # Prepare item_bits once
#     item_bits = [(it, (np.uint64(1) << np.uint64(i))) for it, i in item_to_bit.items()]
#     item_bits = sorted(item_bits, key=lambda x: x[0])

#     rules_3 = []
#     r = 3  # 3-item only

#     for combo in combinations(item_bits, r):
#         ant_items = [c[0] for c in combo]
#         ant_bits = np.uint64(0)
#         for _, b in combo:
#             ant_bits |= b

#         presence = (bits & ant_bits) == ant_bits
#         suppA = presence.mean()

#         # small pruning: if antecedent itself is rarer than MIN_SUPPORT_JOINT_3, joint support cannot exceed it
#         if suppA < MIN_SUPPORT_JOINT_3:
#             continue

#         # Only compute for the "poor" clusters
#         for k in poor_clusters:
#             supp_joint = (presence & cluster_masks[k]).mean()
#             if supp_joint < MIN_SUPPORT_JOINT_3:
#                 continue

#             conf = supp_joint / suppA
#             if conf < MIN_CONFIDENCE_3:
#                 continue

#             lift = conf / p_cluster[k]
#             if lift < MIN_LIFT_3:
#                 continue

#             rules_3.append({
#                 "cluster": k,
#                 "antecedent": " & ".join(ant_items),
#                 "consequent": f"cluster={k}",
#                 "support": supp_joint,
#                 "confidence": conf,
#                 "lift": lift,
#                 "support_antecedent": suppA,
#                 "antecedent_len": 3
#             })

#     rules_3_df = pd.DataFrame(rules_3)
#     print("\n3-item rules found (only for poor clusters):", len(rules_3_df))

#     # 3) Merge: keep your original rules_df (1~2 items), add 3-item rules only for poor clusters
#     rules_df_aug = rules_df.copy()
#     if "antecedent_len" not in rules_df_aug.columns:
#         rules_df_aug["antecedent_len"] = rules_df_aug["antecedent"].apply(lambda s: str(s).count("&") + 1)

#     if len(rules_3_df) > 0:
#         rules_df_aug = pd.concat([rules_df_aug, rules_3_df], ignore_index=True)

#     # 4) Rebuild "Top-N per cluster" result, but ONLY fill gaps for poor clusters
#     #    - For non-poor clusters: keep the original top rules from result
#     #    - For poor clusters: take best rules from augmented pool (lift -> confidence -> support)
#     TOP_N_PER_CLUSTER = TOP_N_PER_CLUSTER  # reuse your existing variable

#     result_new_parts = []

#     # Keep original for non-poor clusters
#     non_poor = [k for k in clusters if k not in poor_clusters]
#     if len(non_poor) > 0:
#         result_new_parts.append(result[result["cluster"].isin(non_poor)].copy())

#     # Recompute top-N for poor clusters from augmented pool
#     for k in poor_clusters:
#         sub = (rules_df_aug[rules_df_aug["cluster"] == k]
#                .sort_values(["lift", "confidence", "support"], ascending=[False, False, False])
#                .head(TOP_N_PER_CLUSTER)
#                .copy())
#         result_new_parts.append(sub)

#     result_aug = pd.concat(result_new_parts, ignore_index=True)
#     result_aug = result_aug.sort_values(["cluster", "lift", "confidence", "support"], ascending=[True, False, False, False])

#     # 5) Save as a NEW file (do not overwrite your original outputs)
#     result_aug.to_csv("cluster_explain_rules_top_augmented.csv", index=False)
#     print("\nSaved: cluster_explain_rules_top_augmented.csv")

#     # Optional: preview
#     print("\nPreview (augmented) top rules per cluster:")
#     print(result_aug[["cluster", "antecedent", "support", "confidence", "lift", "antecedent_len"]]
#           .groupby("cluster")
#           .head(10)
#           .to_string(index=False))

Rules per cluster (1~2 items):
cluster
0     80
1    101
2      8
3      2

Rule-poor clusters to supplement: [3]

3-item rules found (only for poor clusters): 201

Saved: cluster_explain_rules_top_augmented.csv

Preview (augmented) top rules per cluster:
 cluster                                                                                    antecedent  support  confidence      lift  antecedent_len
       0                                         MT_TTC_NET_total_spend=low & store_loyalty_score=high 0.203184    0.998701  3.446200             NaN
       0                            CD_TICKET_UNIQUE_total_transactions=low & store_loyalty_score=high 0.262154    0.994861  3.432949             NaN
       0                                        avg_days_between_visits=low & store_loyalty_score=high 0.263516    0.994640  3.432184             NaN
       0                                                  recency_days=high & store_loyalty_score=high 0.106174    0.993357  3.427756           

In [9]:
import pandas as pd

result = pd.read_csv("cluster_explain_rules_top.csv")

for k in sorted(result["cluster"].unique()):
    sub = result[result["cluster"] == k].sort_values(
        ["lift", "confidence", "support"], ascending=[False, False, False]
    )
    print("\n" + "="*80)
    print(f"Cluster {k} - top rules")
    print(sub[["antecedent","support","confidence","lift"]].head(10).to_string(index=False))



Cluster 0 - top rules
                                                        antecedent  support  confidence     lift
             MT_TTC_NET_total_spend=low & store_loyalty_score=high 0.203184    0.998701 3.446200
CD_TICKET_UNIQUE_total_transactions=low & store_loyalty_score=high 0.262154    0.994861 3.432949
            avg_days_between_visits=low & store_loyalty_score=high 0.263516    0.994640 3.432184
                      recency_days=high & store_loyalty_score=high 0.106174    0.993357 3.427756
            product_diversity_score=low & store_loyalty_score=high 0.250974    0.992749 3.425661
               avg_items_per_basket=low & store_loyalty_score=high 0.178108    0.992135 3.423541
                   avg_basket_value=low & store_loyalty_score=high 0.189956    0.990315 3.417260
                   promo_percentage=low & store_loyalty_score=high 0.246961    0.986770 3.405030
        QTY_PROMO_NAT_nat_promo_qty=low & store_loyalty_score=high 0.251528    0.985571 3.400890
       

In [10]:
df = pd.read_csv("../../customers_with_segments.csv")

FEATURES = [
    "MT_TTC_NET_total_spend",
    "CD_TICKET_UNIQUE_total_transactions",
    "avg_basket_value",
    "avg_items_per_basket",
    "product_diversity_score",
    "promo_percentage",
    "QTY_PROMO_STORE_store_promo_qty",
    "QTY_PROMO_NAT_nat_promo_qty",
    "ID_MAG_TIERS_stores_visited",
    "store_loyalty_score",
    "recency_days",
    "avg_days_between_visits",
]

for c in FEATURES:
    s = pd.to_numeric(df[c], errors="coerce").dropna()
    q33, q66 = s.quantile([1/3, 2/3]).values
    print(f"{c}: low <= {q33:.4g}, mid in ({q33:.4g}, {q66:.4g}], high > {q66:.4g}")


MT_TTC_NET_total_spend: low <= 16.55, mid in (16.55, 40.23], high > 40.23
CD_TICKET_UNIQUE_total_transactions: low <= 1, mid in (1, 1], high > 1
avg_basket_value: low <= 13.61, mid in (13.61, 29.92], high > 29.92
avg_items_per_basket: low <= 5.401, mid in (5.401, 11.64], high > 11.64
product_diversity_score: low <= 6, mid in (6, 22], high > 22
promo_percentage: low <= 0, mid in (0, 0], high > 0
QTY_PROMO_STORE_store_promo_qty: low <= 0, mid in (0, 0], high > 0
QTY_PROMO_NAT_nat_promo_qty: low <= 0, mid in (0, 0], high > 0
ID_MAG_TIERS_stores_visited: low <= 2, mid in (2, 3], high > 3
store_loyalty_score: low <= 0.3333, mid in (0.3333, 0.5], high > 0.5
recency_days: low <= 4, mid in (4, 11], high > 11
avg_days_between_visits: low <= 0, mid in (0, 0], high > 0


In [11]:
import pandas as pd
import numpy as np

# 1) Load the rule file generated in the previous step
rules = pd.read_csv("cluster_explain_rules_top.csv")

# 2) Back-calculate the baseline proportion of each cluster (approx.)
# p(cluster) = confidence / lift
rules["p_cluster_est"] = rules["confidence"] / rules["lift"]

# 3) Estimate how much of each cluster is covered by a rule:
# coverage within cluster = support(A ∧ C) / P(C)
rules["coverage_in_cluster_est"] = rules["support"] / rules["p_cluster_est"]

# 4) Create a business-friendly (plain-language) interpretation of the antecedent
def to_human(row):
    ant = row["antecedent"]

    # In the current binning setup, some features are effectively binary (>0 vs =0)
    ant = ant.replace("promo_percentage=high", "Has promo purchases (>0)")
    ant = ant.replace("promo_percentage=low", "No promo purchases (=0)")
    ant = ant.replace("promo_percentage=mid", "No promo purchases (=0)")

    ant = ant.replace("QTY_PROMO_STORE_store_promo_qty=high", "Has store promo purchases (>0)")
    ant = ant.replace("QTY_PROMO_STORE_store_promo_qty=low", "No store promo purchases (=0)")
    ant = ant.replace("QTY_PROMO_STORE_store_promo_qty=mid", "No store promo purchases (=0)")

    ant = ant.replace("QTY_PROMO_NAT_nat_promo_qty=high", "Has national promo purchases (>0)")
    ant = ant.replace("QTY_PROMO_NAT_nat_promo_qty=low", "No national promo purchases (=0)")
    ant = ant.replace("QTY_PROMO_NAT_nat_promo_qty=mid", "No national promo purchases (=0)")

    ant = ant.replace("CD_TICKET_UNIQUE_total_transactions=high", "Transactions > 1")
    ant = ant.replace("CD_TICKET_UNIQUE_total_transactions=low", "Transactions = 1 (or low)")
    ant = ant.replace("CD_TICKET_UNIQUE_total_transactions=mid", "Transactions = 1 (or low)")

    ant = ant.replace("recency_days=high", "Inactive for a long time (>11 days)")
    ant = ant.replace("recency_days=mid", "Medium recency (5-11 days)")
    ant = ant.replace("recency_days=low", "Recently active (<=4 days)")

    ant = ant.replace("ID_MAG_TIERS_stores_visited=high", "Visited many stores (>3)")
    ant = ant.replace("ID_MAG_TIERS_stores_visited=mid", "Visited a medium number of stores (=3)")
    ant = ant.replace("ID_MAG_TIERS_stores_visited=low", "Visited few stores (<=2)")

    ant = ant.replace("store_loyalty_score=high", "High single-store loyalty (>0.5)")
    ant = ant.replace("store_loyalty_score=mid", "Medium store loyalty (0.333-0.5)")
    ant = ant.replace("store_loyalty_score=low", "Low store loyalty (<=0.333)")

    ant = ant.replace("MT_TTC_NET_total_spend=high", "High total spend")
    ant = ant.replace("MT_TTC_NET_total_spend=mid", "Medium total spend")
    ant = ant.replace("MT_TTC_NET_total_spend=low", "Low total spend")

    ant = ant.replace("avg_basket_value=high", "High average basket value")
    ant = ant.replace("avg_basket_value=mid", "Medium average basket value")
    ant = ant.replace("avg_basket_value=low", "Low average basket value")

    ant = ant.replace("avg_items_per_basket=high", "Large baskets (many items)")
    ant = ant.replace("avg_items_per_basket=mid", "Medium basket size")
    ant = ant.replace("avg_items_per_basket=low", "Small baskets (few items)")

    ant = ant.replace("product_diversity_score=high", "High product diversity")
    ant = ant.replace("product_diversity_score=mid", "Medium product diversity")
    ant = ant.replace("product_diversity_score=low", "Low product diversity")

    ant = ant.replace("avg_days_between_visits=high", "Repeat visits (avg gap > 0)")
    ant = ant.replace("avg_days_between_visits=mid", "Single-day / one-off pattern (avg gap = 0)")
    ant = ant.replace("avg_days_between_visits=low", "Single-day / one-off pattern (avg gap = 0)")

    return ant

rules["business_readable_antecedent"] = rules.apply(to_human, axis=1)

# 5) For each cluster, keep the most representative rules:
# prioritize lift, then coverage within cluster, then confidence
out = []
for k in sorted(rules["cluster"].unique()):
    sub = rules[rules["cluster"] == k].copy()
    sub = sub.sort_values(
        ["lift", "coverage_in_cluster_est", "confidence"],
        ascending=[False, False, False]
    )

    # Approximate baseline share of this cluster (using the first row in the sorted subset)
    sub["cluster_est_size"] = sub["p_cluster_est"].iloc[0]

    out.append(sub.head(10))

report = pd.concat(out, ignore_index=True)

print(report[[
    "cluster",
    "business_readable_antecedent",
    "support",
    "confidence",
    "lift",
    "coverage_in_cluster_est",
    "cluster_est_size"
]].to_string(index=False))

# Save a report-friendly output file
report.to_csv("cluster_profile_report.csv", index=False)
print("\nSaved: cluster_profile_report.csv")

 cluster                                                  business_readable_antecedent  support  confidence     lift  coverage_in_cluster_est  cluster_est_size
       0                            Low total spend & High single-store loyalty (>0.5) 0.203184    0.998701 3.446200                 0.701123          0.289798
       0                  Transactions = 1 (or low) & High single-store loyalty (>0.5) 0.262154    0.994861 3.432949                 0.904610          0.289798
       0 Single-day / one-off pattern (avg gap = 0) & High single-store loyalty (>0.5) 0.263516    0.994640 3.432184                 0.909311          0.289798
       0        Inactive for a long time (>11 days) & High single-store loyalty (>0.5) 0.106174    0.993357 3.427756                 0.366374          0.289798
       0                      Low product diversity & High single-store loyalty (>0.5) 0.250974    0.992749 3.425661                 0.866032          0.289798
       0                  Small baskets 

In [12]:
import glob

files = sorted(glob.glob("association_rules_segment_*.csv"))
if not files:
    print("Didn't find association_rules_segment_*.csv")
else:
    seg_reports = []
    for f in files:
        seg = int(f.split("_")[-1].split(".")[0])  # cluster id
        df = pd.read_csv(f)

        top = df.sort_values(["lift","confidence","support"], ascending=False).head(10).copy()
        top.insert(0, "cluster", seg)
        top.insert(1, "source_file", f)
        seg_reports.append(top)

    seg_report = pd.concat(seg_reports, ignore_index=True)
    seg_report.to_csv("cluster_product_rules_top.csv", index=False)
    print("Saved: cluster_product_rules_top.csv")
    print(seg_report.head(30).to_string(index=False))


Didn't find association_rules_segment_*.csv
